# Laya smoke test

Scores the same HTML corpora as `smoke_test.ipynb`, using a Laya checkpoint instead of
the MarkupLM artifact. `MODEL_DIR` below chooses which one, and the notebook adapts:

| Checkpoint | What it is |
|---|---|
| `artifacts/laya-phishing-v1` | fine-tuned on the same 800-page PhreshPhish sample as MarkupLM `v1` |
| `artifacts/laya_v1` | the published base checkpoint, never trained on phishing |

Both are ModernBERT-large, 421M parameters, 512-token context, and neither sees the DOM
the way MarkupLM does — Laya reads the markup as plain text.

Reference numbers measured earlier on these same corpora:

| | Handwritten | Mendeley AUC | PhreshPhish AUC |
|---|---|---|---|
| Page length alone | — | 0.745 | 0.739 |
| MarkupLM `v1` | 12 / 14 | 0.814 | 0.869 |
| Laya zero-shot | 11 / 14 | 0.724 | not measured |
| Laya fine-tuned | run this notebook | run this notebook | 0.859 (its own split) |

### Reading the fine-tuned model's reported score carefully

`laya-phishing-v1` reports AUC 0.859 on the test split its own training run held out, where
counting characters alone scores 0.823. That is a gain of 0.036 over a baseline that needs
no model at all. MarkupLM `v1`'s 0.869 was measured on a **different** corpus whose length
baseline sits at 0.739, so 0.859 and 0.869 are not directly comparable however similar they
look. Mendeley below is the corpus where both models can be compared on identical pages.


## Setup

`transformers` decorates ModernBERT with `@torch.compile` at import time, and torch refuses
to compile on Python 3.14, so importing the encoder fails before any model is loaded.
`torch.compile` only changes speed, never results, so it is replaced with a pass-through
here. Remove the patch on Python 3.13 or older.

When the checkpoint carries a `metadata.json`, the question is read from it rather than
retyped below. A fine-tuned head answers the exact question it was trained on; asking a
reworded one silently degrades it, and nothing in the output would show why.

To fetch the base checkpoint instead:

```python
from huggingface_hub import snapshot_download
snapshot_download("convaiinnovations/laya", local_dir="artifacts/laya_v1",
                  allow_patterns=["model.safetensors", "encoder/*", "tokenizer/*",
                                  "rl_agent_config.json", "README.md", "eval/results.json"])
```


In [ ]:
import json
import sys
from pathlib import Path

import torch

# Must happen before transformers imports ModernBERT, so keep this cell first.
if sys.version_info >= (3, 14):
    def _passthrough_compile(model=None, **kwargs):
        return (lambda fn: fn) if model is None else model
    torch.compile = _passthrough_compile
    print("torch.compile disabled: unsupported on Python", ".".join(map(str, sys.version_info[:2])))

from bs4 import BeautifulSoup, Comment
from laya import Agent

MODEL_DIR = Path("artifacts/laya-phishing-v1")   # or artifacts/laya_v1 for the base checkpoint

if not MODEL_DIR.is_absolute():
    candidates = [Path.cwd() / MODEL_DIR,
                  Path.cwd() / "ai" / "markuplm" / MODEL_DIR,
                  Path.cwd() / "markuplm" / MODEL_DIR]
    MODEL_DIR = next((p for p in candidates if (p / "model.safetensors").is_file()), candidates[0])
MODEL_DIR = MODEL_DIR.resolve()
if not (MODEL_DIR / "model.safetensors").is_file():
    raise FileNotFoundError(f"No Laya checkpoint at {MODEL_DIR}. See the download snippet above.")

metadata_path = MODEL_DIR / "metadata.json"
metadata = json.loads(metadata_path.read_text(encoding="utf-8")) if metadata_path.is_file() else {}
FINE_TUNED = bool(metadata.get("train_n"))

agent = Agent(str(MODEL_DIR))
print("Model:", MODEL_DIR.name)
if FINE_TUNED:
    scores = metadata.get("test_metrics", {})
    print(f"fine-tuned on {metadata['train_n']} pages, best epoch {metadata.get('best_epoch')}")
    print(f"  its own test split: AUC {scores.get('auc')}, accuracy {scores.get('accuracy')}")
    print(f"  length-only baseline on that split: {metadata.get('baselines', {}).get('length_only_auc_test')}")
else:
    print("base checkpoint: nothing about phishing was trained into it")

## The question, and what the model is shown

For a fine-tuned checkpoint the question is not a free choice: the head was trained against
one specific wording, so it is read back from `metadata.json`. For the base checkpoint the
wording is the measurement, and changing it changes the numbers.

Both models get HTML cleaned the way MarkupLM `v1` cleans it — scripts, styles and comments
removed, 300k characters at most — so no model is handed a different page.

Cleaned markup beats visible text alone, measured on the handwritten examples with the base
checkpoint: 11/14 against 10/14, and a wider gap between the classes (0.824 vs 0.295 with
markup, 0.952 vs 0.456 with text). Tags cost tokens but carry the `<form>` and
`<input type=password>` elements that give a credential trap away.


In [ ]:
# A fine-tuned head answers the question it was trained on; take it from metadata when present.
trained_question = metadata.get("question") or {}
INSTRUCTIONS = trained_question.get("ins") or (
    "Is this web page a phishing page that impersonates a real brand to steal user credentials?")
QUESTION = {"phishing": {"type": "noul", "instructions": INSTRUCTIONS}}
print("question:", INSTRUCTIONS)
print("source  :", "metadata.json (matches training)" if trained_question else "notebook default")

# Preprocessing follows the checkpoint's own record when it has one.
_prep = metadata.get("preprocessing") or {}
PARSE_CHAR_BUDGET = int(_prep.get("parse_char_budget", 300_000))
STRIP_TAGS = list(_prep.get("strip_tags", ["script", "style", "noscript", "template"]))
THRESHOLD = float((metadata.get("inference") or {}).get("operating_threshold", 0.5))
print(f"cleaning: strip {STRIP_TAGS}, budget {PARSE_CHAR_BUDGET:,} chars, threshold {THRESHOLD}")


def clean_html(html):
    soup = BeautifulSoup(html[:PARSE_CHAR_BUDGET], "html.parser")
    for element in soup.find_all(STRIP_TAGS):
        element.decompose()
    for comment in soup.find_all(string=lambda text: isinstance(text, Comment)):
        comment.extract()
    return str(soup)


def predict_html(html):
    """Mirrors predict_html in smoke_test.ipynb so the two notebooks stay comparable."""
    if not isinstance(html, str):
        raise TypeError("html must be a string")
    cleaned = clean_html(html)
    if not cleaned.strip():
        raise ValueError("No usable markup")
    result = agent.predict(cleaned, QUESTION)
    score = float(result["answers"]["phishing"]["noul"])
    return {
        "prediction": "phishing" if score >= THRESHOLD else "benign",
        "phishing_score": score,
        "benign_score": 1.0 - score,
        "threshold": THRESHOLD,
        "html_chars": len(html),
        "input_tokens": int(result.get("usage", {}).get("input_tokens", 0)),
    }


_sample = Path("examples/phishing/klik_bca.html")
demo = predict_html(_sample.read_text(encoding="utf-8") if _sample.exists()
                    else "<html><body><form><input type=password></form></body></html>")
print(json.dumps(demo, indent=2))

## Handwritten diagnostics

The same 14 hand-labelled files `smoke_test.ipynb` uses, so the two notebooks can be read
side by side. MarkupLM `v1` gets 12 of 14. Fourteen examples decide nothing on their own;
they only show whether the model is behaving sanely before the larger corpus runs.


In [ ]:
EXAMPLES_DIR = Path("examples")
if not EXAMPLES_DIR.is_absolute():
    candidates = [Path.cwd() / EXAMPLES_DIR,
                  Path.cwd() / "ai" / "markuplm" / EXAMPLES_DIR,
                  Path.cwd() / "markuplm" / EXAMPLES_DIR]
    EXAMPLES_DIR = next((p for p in candidates
                         if (p / "benign").is_dir() and (p / "phishing").is_dir()), candidates[0])
EXAMPLES_DIR = EXAMPLES_DIR.resolve()

# The folder sets the expected label; the model never gets a say in its own ground truth.
example_files = [(path, label)
                 for folder, label in (("benign", "benign"), ("phishing", "phishing"))
                 for path in sorted((EXAMPLES_DIR / folder).glob("*.html"))]
if not example_files:
    raise FileNotFoundError(f"No diagnostic HTML under {EXAMPLES_DIR}")

passed = 0
for path, expected in example_files:
    name = path.relative_to(EXAMPLES_DIR).as_posix()
    try:
        result = predict_html(path.read_text(encoding="utf-8", errors="replace"))
        ok = result["prediction"] == expected
        passed += ok
        print(("PASS" if ok else "FAIL"), f"{name:42}",
              f"score={result['phishing_score']:.3f} tokens={result['input_tokens']}")
    except (OSError, ValueError, RuntimeError) as exc:
        print("ERROR", name, exc)
print(f"\n{passed}/{len(example_files)} handwritten examples passed "
      f"(MarkupLM v1 scores 12/14 on these)")

## Cross-dataset run

The 300 Mendeley pages built by `scripts/download_mendeley_html.py`. Laya never saw this
corpus, and it never saw PhreshPhish either, so unlike MarkupLM `v1` there is no home
advantage to lose — whatever it scores here is simply what a general model does cold.

The trivial baseline is recomputed on the same pages. Anything at or below it means the
answer carries no information beyond "shorter pages look suspicious".


In [ ]:
import time

CORPUS_DIR = EXAMPLES_DIR / "mendeley"
if not (CORPUS_DIR / "benign").is_dir() or not (CORPUS_DIR / "phishing").is_dir():
    raise FileNotFoundError(
        f"No corpus at {CORPUS_DIR}. Build one first:\n"
        "    python scripts/download_mendeley_html.py --per-class 150")

corpus_files = [(path, label)
                for folder, label in (("benign", "benign"), ("phishing", "phishing"))
                for path in sorted((CORPUS_DIR / folder).glob("*.html"))]
print(f"{CORPUS_DIR.name}: {len(corpus_files)} pages (roughly {len(corpus_files) * 1.5 / 60:.0f} min)")


def roc_auc(positive_scores, negative_scores):
    """Rank-based AUC, tie-corrected. 0.5 is a coin flip, 1.0 is perfect."""
    if not positive_scores or not negative_scores:
        return float("nan")
    merged = sorted([(s, 1) for s in positive_scores] + [(s, 0) for s in negative_scores])
    ranks, index = {}, 0
    while index < len(merged):
        end = index
        while end + 1 < len(merged) and merged[end + 1][0] == merged[index][0]:
            end += 1
        average = (index + end) / 2 + 1
        for position in range(index, end + 1):
            ranks[position] = average
        index = end + 1
    rank_sum = sum(ranks[i] for i, (_, label) in enumerate(merged) if label == 1)
    n_pos, n_neg = len(positive_scores), len(negative_scores)
    return (rank_sum - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


model_scores = {"phishing": [], "benign": []}
length_scores = {"phishing": [], "benign": []}
counts = {"tp": 0, "fp": 0, "tn": 0, "fn": 0}
failures = []
started = time.time()

for position, (path, expected) in enumerate(corpus_files, start=1):
    try:
        result = predict_html(path.read_text(encoding="utf-8", errors="replace"))
    except (OSError, ValueError, RuntimeError) as exc:
        failures.append(path.name)
        continue
    model_scores[expected].append(result["phishing_score"])
    # Shorter means more suspicious, negated so higher still means "more phishing".
    length_scores[expected].append(-result["html_chars"])
    if result["phishing_score"] >= THRESHOLD:
        counts["tp" if expected == "phishing" else "fp"] += 1
    else:
        counts["fn" if expected == "phishing" else "tn"] += 1
    if position % 50 == 0:
        print(f"  {position}/{len(corpus_files)}  {time.time() - started:.0f}s", flush=True)

scored = sum(counts.values())
if scored == 0:
    raise RuntimeError("Every page failed; nothing to report.")

accuracy = (counts["tp"] + counts["tn"]) / scored
precision = counts["tp"] / (counts["tp"] + counts["fp"]) if counts["tp"] + counts["fp"] else 0.0
recall = counts["tp"] / (counts["tp"] + counts["fn"]) if counts["tp"] + counts["fn"] else 0.0
laya_auc = roc_auc(model_scores["phishing"], model_scores["benign"])
length_auc = roc_auc(length_scores["phishing"], length_scores["benign"])

print(f"\nscored {scored} pages in {time.time() - started:.0f}s, {len(failures)} unusable")
print(f"accuracy  {accuracy:.3f}   precision {precision:.3f}   recall {recall:.3f}")
print("\nconfusion          predicted phishing   predicted benign")
print(f"  truly phishing   {counts['tp']:>18}   {counts['fn']:>16}")
print(f"  truly benign     {counts['fp']:>18}   {counts['tn']:>16}")
label = "Laya, fine-tuned" if FINE_TUNED else "Laya, zero-shot"
print("\n                                AUC")
print(f"  {label:26}  {laya_auc:.3f}")
print(f"  {'page length alone':26}  {length_auc:.3f}   <- the floor worth clearing")
print(f"  {'MarkupLM v1, fine-tuned':26}  0.814   <- same pages, 756-page sample")
if FINE_TUNED:
    print(f"  {'Laya before fine-tuning':26}  0.724   <- same pages, same question")

if laya_auc <= length_auc:
    print("\nThis checkpoint adds nothing over counting characters on these pages.")
elif FINE_TUNED:
    gain = laya_auc - 0.724
    verdict = ("ahead of MarkupLM v1" if laya_auc > 0.814 else
               f"behind MarkupLM v1 by {0.814 - laya_auc:.3f}")
    print(f"\nFine-tuning moved it {gain:+.3f} AUC against the same pages and the same question, "
          f"and it lands {verdict}. This corpus is the fair comparison: both models are scored "
          "on identical pages, and neither trained on any of them.")
elif laya_auc < 0.814:
    print(f"\nBelow the fine-tuned model by {0.814 - laya_auc:.3f} AUC, which is the price of "
          "never having seen a phishing page. It still beats the trivial baseline.")
else:
    print("\nA model never trained on phishing matched or beat the fine-tuned one, which "
          "says more about how little the fine-tuning data taught than about Laya.")

if failures:
    print(f"\nunusable: {failures[:5]}")
